# Privacy and Governance Analysis

This notebook analyzes privacy risks and governance issues in the NovaCred credit application dataset.

While earlier stages of the project focused on data quality and bias detection, this section examines how personal data is handled and what governance mechanisms should be implemented to ensure responsible data use.

The goal of this analysis is to:

- identify personal and sensitive data present in the dataset
- evaluate potential privacy risks
- assess whether the dataset follows data minimization principles
- discuss governance practices that support responsible AI systems

These considerations are especially important in financial contexts such as credit approval systems, where personal and behavioral data must be handled carefully to protect individuals' privacy.

## 1. Loading the Cleaned Dataset

To ensure consistency across the project, we use the cleaned dataset generated during the data quality stage. 

This dataset consolidates the original application information with spending behavior data and applies the preprocessing rules defined earlier in the pipeline.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/cleaned_credit_applications.csv')

print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

print("\nColumns in dataset:\n")
for col in df.columns:
    print("-", col)

df.head()

Dataset shape: 500 rows x 35 columns

Columns in dataset:

- _id
- processing_timestamp
- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- applicant_info.gender
- applicant_info.date_of_birth
- applicant_info.zip_code
- financials.annual_income
- financials.credit_history_months
- financials.debt_to_income
- financials.savings_balance
- decision.loan_approved
- decision.rejection_reason
- loan_purpose
- decision.interest_rate
- decision.approved_amount
- notes
- spending_Adult_Entertainment
- spending_Alcohol
- spending_Dining
- spending_Education
- spending_Entertainment
- spending_Fitness
- spending_Gambling
- spending_Groceries
- spending_Healthcare
- spending_Insurance
- spending_Rent
- spending_Shopping
- spending_Transportation
- spending_Travel
- spending_Utilities
- flag_duplicate_ssn


,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities,flag_duplicate_ssn
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-09-03,10036.0,73000.0,...,0,0,0,0,790,480,0,0,0,False
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,NaN,10032.0,78000.0,...,0,0,243,0,608,0,0,0,0,False
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,NaN,10075.0,61000.0,...,0,0,0,0,109,0,0,0,0,False
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,NaN,10077.0,103000.0,...,0,0,0,0,0,0,0,0,0,False
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,NaN,10080.0,57000.0,...,0,0,0,0,0,0,0,0,0,False


### Initial Observations:

The cleaned dataset contains 500 records and 35 variables, combining applicant information, financial indicators, and spending behavior features.

Some columns contain highly sensitive personal information, such as names, emails, Social Security Numbers (SSN), IP addresses, and dates of birth. These variables represent direct identifiers and highlight the importance of applying strong privacy and governance controls before using the data for analytical purposes.

## 2. Identifying Personal Data (PII)

Before we do any fairness or modelling work, we need to be clear about what personal data exists in the dataset.

For this project, we separate attributes into:

- **Direct identifiers (PII)**: values that can identify someone on their own (e.g., name, email, SSN).
- **Quasi-identifiers**: values that may not identify someone alone, but can become identifying when combined (e.g., gender + ZIP + income).

This matters because privacy risk is often driven by **combinations** of “innocent-looking” fields rather than a single column.

In [6]:
direct_identifiers = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth"
]

quasi_identifiers = [
    "applicant_info.gender",
    "applicant_info.zip_code",
    "financials.annual_income"
]

print("Direct identifiers detected:\n")

for col in direct_identifiers:
    if col in df.columns:
        print("✔", col)

print("\nQuasi-identifiers detected:\n")

for col in quasi_identifiers:
    if col in df.columns:
        print("✔", col)

Direct identifiers detected:

✔ applicant_info.full_name
✔ applicant_info.email
✔ applicant_info.ssn
✔ applicant_info.ip_address
✔ applicant_info.date_of_birth

Quasi-identifiers detected:

✔ applicant_info.gender
✔ applicant_info.zip_code
✔ financials.annual_income


### Interpretation:

The cleaned dataset still contains several **direct identifiers** (full name, email, SSN, IP address, and date of birth). This is a high privacy risk, especially in a credit scoring context, because these fields can directly identify individuals.

In addition, variables such as gender, zip code, and annual income act as **quasi-identifiers**. Even if direct identifiers were removed, combinations of these attributes could still enable re-identification. For this reason, the dataset should be access-restricted and direct identifiers should be removed or pseudonymized before any analysis or modeling.

## 3. Re-identification Risk (Quasi-identifiers)

Even if we removed direct identifiers, individuals could still be re-identified through **unique combinations** of quasi-identifiers.

To estimate this, we look at the group size (*k*) for each combination of:
**gender + ZIP code + annual income**.

If many records have **k = 1**, the dataset is highly re-identifiable.

In [15]:
qi = ["applicant_info.gender", "applicant_info.zip_code", "financials.annual_income"]

# group size per quasi-identifier combination
group_sizes = df.groupby(qi).size().rename("k").reset_index()

# attach k back to each row (so we can count how many rows have k=1 etc.)
df_k = df.merge(group_sizes, on=qi, how="left")

print("Total records:", len(df_k))
print("Unique quasi-identifier combinations:", group_sizes.shape[0])
print("Estimated re-identification ratio (unique combos / records):", round(group_sizes.shape[0] / len(df_k), 3))

print("\nHow many records fall into each k-anonymity level?")
print(df_k["k"].value_counts().sort_index().head(10))

share_k1 = (df_k["k"] == 1).mean()
share_k2 = (df_k["k"] == 2).mean()
share_kge3 = (df_k["k"] >= 3).mean()

print(f"\nShare of records with k=1 (unique): {share_k1:.1%}")
print(f"Share of records with k=2: {share_k2:.1%}")
print(f"Share of records with k>=3: {share_kge3:.1%}")

Total records: 500
Unique quasi-identifier combinations: 493
Estimated re-identification ratio (unique combos / records): 0.986

How many records fall into each k-anonymity level?
k
1.0    487
2.0     12
Name: count, dtype: int64

Share of records with k=1 (unique): 97.4%
Share of records with k=2: 2.4%
Share of records with k>=3: 0.0%


### Interpretation:

The results indicate an extremely high re-identification risk in the dataset.

Out of 500 records, **493 quasi-identifier combinations are unique**, resulting in a re-identification ratio of **0.986**. This means that almost every applicant has a distinct combination of gender, ZIP code, and annual income.

The k-anonymity analysis confirms this pattern: **97.4% of records have k = 1**, meaning that those individuals are unique in the dataset with respect to the selected quasi-identifiers. Only a small fraction of records share the same profile (k = 2), and none reach k ≥ 3.

In practice, this means that even if direct identifiers such as names or SSNs were removed, most individuals could still be singled out using combinations of seemingly harmless attributes. If external information were available (for example public demographic data), the risk of linking records back to real individuals would be substantial.

From a governance perspective, this suggests that additional privacy protections are necessary. Possible mitigation strategies include **generalizing quasi-identifiers** (e.g., using income ranges instead of exact values), **reducing geographic precision**, or applying formal anonymization techniques before the dataset is used for analysis or shared internally.

## 4. Data Minimization Assessment (GDPR)

A key GDPR principle is **data minimization**: we should only use data that is necessary for the stated purpose.

In this project, the purpose is to analyze credit approval outcomes and governance risks — not to contact applicants or uniquely identify them.

So we flag the following fields as **non-essential for modelling/analysis** and high-risk from a privacy standpoint.

In [16]:
non_essential_fields = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "notes",
]

print("Fields that should not be used for modelling:\n")
for col in non_essential_fields:
    if col in df.columns:
        print("-", col)

Fields that should not be used for modelling:

- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- notes


### Interpretation

The results confirm that several attributes in the dataset are direct personal identifiers, including the applicant’s full name, email address, Social Security Number, and IP address. These variables are not required to analyze credit approval outcomes and therefore should not be used in modelling or analytical tasks.

Keeping these identifiers in analytical datasets increases privacy risks without adding predictive value. In particular, highly sensitive attributes such as SSNs or IP addresses could directly expose individuals if the dataset were accessed or shared beyond its intended environment.

From a GDPR perspective, retaining these fields would violate the **data minimization principle**, which requires organizations to process only the data that is strictly necessary for the stated purpose. Since the goal of this project is to analyze approval decisions and governance risks—not to identify or contact applicants—these identifiers should be removed or pseudonymized before any analytical use of the data.

In practice, this means that raw identifiers should be excluded from modelling datasets and replaced with pseudonymous IDs where record-level tracking is required.

This approach also aligns with best practices in privacy-preserving data analysis, where analytical datasets are separated from datasets containing personally identifiable information.

## 5. Privacy Demonstration: Pseudonymization (Direct Identifiers)

To reduce exposure, we pseudonymize direct identifiers instead of keeping them in raw form.

Important nuance: **pseudonymization is not anonymization**.  
The dataset can still be considered personal data under GDPR if re-linking is possible (e.g., through a mapping table or keys).

In [17]:
import hashlib
import os

# Simple salted hashing helper (good enough for a demo; in real systems we'd use managed secrets)
SALT = os.environ.get("NOVACRED_SALT", "novacred_demo_salt_change_me")

def salted_hash(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    return hashlib.sha256((SALT + x).encode("utf-8")).hexdigest()

df_priv = df.copy()

# Create pseudonymous versions
df_priv["pii.email_hash"] = df_priv["applicant_info.email"].apply(salted_hash)
df_priv["pii.ssn_hash"] = df_priv["applicant_info.ssn"].apply(salted_hash)

# Optional: a stable pseudo ID (based on SSN hash for the demo)
df_priv["applicant_pseudo_id"] = "APP_" + df_priv["pii.ssn_hash"].str[:12]

# Drop raw identifiers from the working dataset
drop_cols = ["applicant_info.full_name", "applicant_info.email", "applicant_info.ssn", "applicant_info.ip_address"]
df_priv = df_priv.drop(columns=[c for c in drop_cols if c in df_priv.columns])

df_priv[["applicant_pseudo_id", "pii.email_hash", "pii.ssn_hash"]].head()

,applicant_pseudo_id,pii.email_hash,pii.ssn_hash
0,APP_ca867004a4b2,b632466d9d06b1a8a6574f9a7c2001133f961628378483...,ca867004a4b28dd21dd306c5a665375ab13e66fda3b12e...
1,APP_2d8627ec48e9,eb82075e50646aa9a7618509d8619bdcc3d29ff544e8ae...,2d8627ec48e985217c3c39d551888001ab1334a8a85984...
2,APP_b0569893aabe,92b675dd2e2e57ff0a2fc7dd67e4a0f0bf264ebce0df01...,b0569893aabeaa023c4dc31c08ea90cc0dd68a5c493623...
3,APP_efb34b2be0ff,433e6e746d188c9eef54c4bf30b58daf5cf0a74fdfa892...,efb34b2be0ff03c1991b67cd05dcb77731edeb75707ce5...
4,APP_26718f63369f,9b237b3cd9aa8f5dc882e70527a360c643881b73fd6ff4...,26718f63369fd5d10903074a0e4eff6925e14fb8cc1007...


### Interpretation:

The transformation above demonstrates a basic pseudonymization approach for sensitive identifiers. Instead of storing raw personal information such as email addresses and Social Security Numbers, the values are converted into salted cryptographic hashes.

This significantly reduces the risk of accidental exposure because the original identifiers cannot be directly read from the dataset. At the same time, the generated `applicant_pseudo_id` allows records to remain traceable within the dataset, which is useful for analytical tasks that require consistent record-level tracking.

However, it is important to note that pseudonymization does not fully anonymize the data. If the hashing method or the salt were known, or if a separate mapping table existed, the original identifiers could theoretically be reconstructed or linked back to individuals. For this reason, the dataset would still be considered **personal data under GDPR**, and appropriate governance controls (such as restricted access and key management) would still be required.

This approach follows common privacy-preserving data practices, where analytical datasets contain pseudonymous identifiers while direct identifiers are stored separately under stricter access controls.

## 6. Reducing Re-identification Risk (Generalization of Quasi-identifiers)

To reduce re-identification risk, we generalize quasi-identifiers:

- **ZIP code → ZIP prefix** (less granular location)
- **date of birth → age band** (keeps signal while removing exact DOB)

Then we recompute the k-anonymity distribution to check if uniqueness decreases.

In [18]:
from datetime import datetime

df_anon = df_priv.copy()

# ZIP generalization: keep only first 3 digits (example). Adjust based on what makes sense in your data.
def zip_prefix(z, n=3):
    if pd.isna(z):
        return np.nan
    z = str(z)
    return z[:n] + ("*" * max(0, len(z) - n))

# DOB -> age band
def age_band(dob):
    if pd.isna(dob):
        return np.nan
    dob = pd.to_datetime(dob, errors="coerce")
    if pd.isna(dob):
        return np.nan
    age = int((pd.Timestamp("today") - dob).days / 365.25)
    # simple bands
    bins = [0, 25, 35, 45, 55, 65, 120]
    labels = ["<25", "25–34", "35–44", "45–54", "55–64", "65+"]
    return pd.cut([age], bins=bins, labels=labels, right=False)[0]

df_anon["applicant_info.zip_code_generalized"] = df_anon["applicant_info.zip_code"].apply(zip_prefix)
df_anon["applicant_info.age_band"] = df_anon["applicant_info.date_of_birth"].apply(age_band)

# Drop raw DOB and raw ZIP (keep generalized versions only)
df_anon = df_anon.drop(columns=[c for c in ["applicant_info.date_of_birth", "applicant_info.zip_code"] if c in df_anon.columns])

# Recompute k-anonymity using generalized quasi-identifiers
qi_gen = ["applicant_info.gender", "applicant_info.zip_code_generalized", "financials.annual_income", "applicant_info.age_band"]

group_sizes_gen = df_anon.groupby(qi_gen).size().rename("k").reset_index()
df_anon_k = df_anon.merge(group_sizes_gen, on=qi_gen, how="left")

print("After generalization:")
print("Unique quasi-identifier combinations:", group_sizes_gen.shape[0])
print("Share of records with k=1:", (df_anon_k["k"] == 1).mean().round(3))
print("Share of records with k>=3:", (df_anon_k["k"] >= 3).mean().round(3))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
After generalization:
Unique quasi-identifier combinations: 135
Share of records with k=1: 0.254
Share of records with k>=3: 0.0


### Interpretation:

After generalizing the quasi-identifiers, the number of unique combinations in the dataset decreases substantially. The number of distinct quasi-identifier profiles drops from nearly the entire dataset to **135 combinations**, and the share of records with k = 1 decreases to approximately **25.4%**.

This indicates that generalizing sensitive attributes such as ZIP code and date of birth can significantly reduce the likelihood that individuals can be uniquely identified through combinations of quasi-identifiers.

However, the results also show that some re-identification risk still remains. A portion of records continue to have k = 1, meaning that certain individuals remain unique even after the transformation. This highlights an important trade-off in privacy-preserving data analysis: increasing privacy protection often requires reducing data granularity.

From a governance perspective, this suggests that additional techniques—such as further generalization, suppression of rare records, or formal anonymization methods—may be necessary before sharing the dataset more broadly or using it in less controlled environments.

These results illustrate the core principle behind k-anonymity: reducing the uniqueness of records makes it harder to link individuals to external information sources.

## 7. GDPR Mapping and Governance Recommendations

Based on the dataset contents and the transformations above, we map the main GDPR principles to practical controls:

### Lawful basis (GDPR)
For a credit-risk context, a plausible lawful basis is **legitimate interests** (risk management) or **contract necessity** (processing applications).  
However, the dataset as provided does not include any explicit consent or lawful-basis tracking, which is a governance gap.

### Data minimization
We removed direct identifiers (name, email, SSN, IP) from the analysis dataset and avoided using free-text notes for modelling.

### Storage limitation
A clear retention policy should exist (e.g., keep application-level data only as long as necessary for audits/regulatory needs).  
Older records should be deleted or fully anonymized.

### Integrity & confidentiality (security controls)
Recommended controls include:
- role-based access to raw PII (need-to-know)
- audit logs for dataset access and model decisions
- encryption at rest + in transit
- separate storage for any re-linking keys (if pseudonymization is reversible)

### Data subject rights (including Article 17 “right to erasure”)
If we keep a re-linking mechanism (e.g., mapping table), we must be able to locate and delete an individual’s data on request.  
If we fully anonymize data (no re-linking possible), Article 17 requests may no longer apply to the anonymized extract — but the organization must be able to justify that anonymization is robust.

### Governance across the data lifecycle

From a governance perspective, personal data should be managed across the entire data lifecycle — from collection and storage to processing, sharing, and eventual deletion.

In this project, governance interventions were applied mainly at the processing stage. Sensitive attributes were pseudonymized and quasi-identifiers were generalized before the dataset was used for analysis.

These transformations reduce re-identification risk while still preserving analytical value, aligning the dataset with responsible data governance practices.

## 8. Governance Controls Summary

Based on the analysis conducted in this notebook, several governance measures should be implemented to ensure that the credit application dataset is processed in a compliant and responsible way.

First, direct identifiers such as names, emails, SSNs, and IP addresses should never be used in analytical datasets. These fields should be removed or pseudonymized before data is accessed by analysts or used in modelling pipelines.

Second, organizations should implement role-based access controls so that only authorized personnel can access raw personal data. Analysts and data scientists should typically work with pseudonymized datasets.

Third, privacy-preserving transformations such as generalization or aggregation should be applied when datasets are shared internally or externally in order to reduce re-identification risks.

Fourth, organizations should establish clear data governance roles, including a data owner responsible for accountability, data stewards responsible for monitoring data quality and compliance, and data custodians responsible for infrastructure security and access management.

Finally, since credit decision systems fall under high-risk AI use cases in the EU AI Act, organizations must maintain clear documentation, auditability, and human oversight for automated decision systems.

Together, these governance practices help ensure that credit risk models are developed in a way that aligns with GDPR principles and modern AI governance standards.

Since credit decision systems influence individuals’ access to financial services, they fall under **high-risk AI systems** in the EU AI Act framework. This means organizations must ensure strong governance controls, including documentation of model development, monitoring of potential bias, human oversight of automated decisions, and clear accountability for how data is collected and processed.

## 9. Output: Privacy-Preserving Dataset for Analysis

Finally, we save a privacy-safer version of the dataset for downstream analysis and sharing within the team.

In [20]:
import os

os.makedirs("../data/processed", exist_ok=True)
out_path = "../data/processed/privacy_safe_credit_applications.csv"
df_anon.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", df_anon.shape)

Saved: ../data/processed/privacy_safe_credit_applications.csv
Shape: (500, 34)


## 10. Governance Audit Conclusion

This analysis highlights that the original credit application dataset contains multiple direct identifiers and quasi-identifiers that create significant privacy risks. Without appropriate safeguards, individuals could potentially be re-identified through combinations of demographic and financial attributes.

Through the application of privacy-preserving techniques—including pseudonymization, removal of direct identifiers, and generalization of quasi-identifiers—the dataset can be transformed into a safer analytical dataset while still preserving its usefulness for modelling and governance analysis.

From a governance perspective, this exercise demonstrates the importance of **privacy-by-design principles**, where personal data protection is integrated directly into data pipelines rather than addressed after models are built.

Implementing strong governance controls, clear accountability roles, and privacy-aware data engineering practices is essential for organizations deploying AI systems in sensitive domains such as credit decision-making.